# 02 — Procesamiento Distribuido con Dask (Cloud — GCS)

Versión cloud del notebook Dask. Los CSVs se leen directamente desde **Google Cloud Storage** (`gs://big-data-proyecto-parcial/raw/`) en lugar del disco local. El cómputo corre en el nodo Dataproc mediante `LocalCluster`.

**Flujo:** GCS raw/ (2017–2025) → Dask LocalCluster (Dataproc node) → resultados por era COVID → BigQuery

| # | Operación | Descripción | Tabla BigQuery |
|---|-----------|-------------|----------------|
| 1 | **Create** | Distribución horaria de crímenes por era COVID | `hourly_distribution` |
| 2 | **Read**   | Top 10 tipos de crimen por año (2017–2025) | — (exploración) |
| 3 | **Read**   | Tasa de arresto por tipo de crimen y era | `arrest_rate_by_type` |
| 4 | **Update** | Añadir `covid_era` + `severity_category` | `severity_distribution` |
| 5 | **Delete** | Eliminar registros sin coordenadas geográficas | — (limpieza) |

In [1]:
import subprocess
subprocess.run(['pip', 'install', 'gcsfs', 'pandas-gbq', '--quiet'], check=True)
print('Dependencias cloud OK')

Dependencias cloud OK


In [2]:
import dask.dataframe as dd
from dask.distributed import Client, LocalCluster
import pandas as pd
import time

PROJECT_ID = 'my-first-project-492901'
DATASET_ID = 'chicago_crimes_results'
GCS_BUCKET = 'gs://big-data-proyecto-parcial/raw'

YEARS_ANALYSIS = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
ERA_MAP = {2017:'PRE', 2018:'PRE', 2019:'PRE',
           2020:'DURANTE', 2021:'DURANTE', 2022:'DURANTE',
           2023:'POST', 2024:'POST', 2025:'POST'}

def to_bigquery(df: pd.DataFrame, table_name: str, if_exists: str = 'replace') -> None:
    df.to_gbq(destination_table=f'{DATASET_ID}.{table_name}',
              project_id=PROJECT_ID, if_exists=if_exists, progress_bar=False)
    print(f'  → BigQuery: {PROJECT_ID}.{DATASET_ID}.{table_name}  ({len(df):,} filas)')

cluster = LocalCluster()
client  = Client(cluster)
print(client)
print(f'Dashboard Dask: {client.dashboard_link}')

<Client: 'tcp://127.0.0.1:43949' processes=4 threads=4, memory=15.63 GiB>
Dashboard Dask: http://127.0.0.1:8787/status


In [3]:
# ── Carga desde GCS (cambio clave vs. versión local) ─────────────────────────
files = [f'{GCS_BUCKET}/Chicago_Crimes_{y}.csv' for y in YEARS_ANALYSIS]

dtypes = {
    'unique_key': 'Int64', 'case_number': 'object', 'block': 'object',
    'iucr': 'object', 'primary_type': 'object', 'description': 'object',
    'location_description': 'object', 'beat': 'Int64', 'district': 'Int64',
    'ward': 'Int64', 'community_area': 'Int64', 'fbi_code': 'object',
    'x_coordinate': 'Int64', 'y_coordinate': 'Int64', 'year': 'Int64',
    'latitude': 'float64', 'longitude': 'float64', 'location': 'object',
}

t0 = time.time()
df = dd.read_csv(
    files,
    dtype=dtypes,
    parse_dates=['date', 'updated_on'],
    assume_missing=True,
    storage_options={"token": "google_default"},
)
total = len(df)
print(f'Fuente:             GCS — {GCS_BUCKET}/')
print(f'Registros cargados: {total:,}  ({time.time()-t0:.1f}s)')
print(f'Años:               {YEARS_ANALYSIS}')
print(f'Particiones Dask:   {df.npartitions}')

Fuente:             GCS — gs://big-data-proyecto-parcial/raw/
Registros cargados: 2,072,943  (17.6s)
Años:               [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Particiones Dask:   9


---
## CRUD 1 — CREATE: Distribución horaria por era COVID

**Operación:** Se crea la columna `hour_of_day` y se genera una tabla que muestra, para cada hora del día, cuántos crímenes y arrestos ocurrieron en cada era (PRE / DURANTE / POST).

**Fuente de datos:** `gs://big-data-proyecto-parcial/raw/` — lectura directa desde GCS sin descarga previa al disco.

In [4]:
df['hour_of_day'] = df['date'].dt.hour
df['covid_era']   = df['year'].map(ERA_MAP, meta=('covid_era', 'object'))

hourly = (
    df.groupby(['hour_of_day', 'covid_era'])
      .agg({'unique_key': 'count', 'arrest': 'sum'})
      .rename(columns={'unique_key': 'total_crimes', 'arrest': 'arrests'})
      .compute()
      .reset_index()
      .sort_values(['hour_of_day', 'covid_era'])
)
hourly['arrest_rate_pct'] = (hourly['arrests'] / hourly['total_crimes'] * 100).round(2)

night = hourly[(hourly['hour_of_day'] >= 22) | (hourly['hour_of_day'] <= 5)]
night_by_era = night.groupby('covid_era')['total_crimes'].sum()
total_by_era = hourly.groupby('covid_era')['total_crimes'].sum()
print('Índice de criminalidad nocturna por era:')
for era in ['PRE', 'DURANTE', 'POST']:
    idx = night_by_era.get(era, 0) / total_by_era.get(era, 1) * 100
    print(f'  {era:<8}: {idx:.2f}%')

to_bigquery(hourly[['hour_of_day', 'covid_era', 'total_crimes', 'arrests', 'arrest_rate_pct']], 'hourly_distribution')

Índice de criminalidad nocturna por era:
  PRE     : 24.42%
  DURANTE : 28.01%
  POST    : 27.99%


/tmp/ipykernel_38233/2663512432.py:16: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  df.to_gbq(destination_table=f'{DATASET_ID}.{table_name}',


  → BigQuery: my-first-project-492901.chicago_crimes_results.hourly_distribution  (72 filas)


---
## CRUD 2 — READ: Top 10 tipos de crimen por año (2017–2025)

In [5]:
top_by_year = (
    df.groupby(['year', 'primary_type'])
      .size()
      .reset_index()
      .rename(columns={0: 'count'})
      .compute()
      .sort_values(['year', 'count'], ascending=[True, False])
)
top10 = top_by_year.groupby('year').head(10).reset_index(drop=True)

print('Top 5 crímenes por año:')
for y in YEARS_ANALYSIS:
    era = ERA_MAP[y]
    top5 = top10[top10['year'] == y].head(5)[['primary_type','count']]
    print(f'  {y} [{era}]: ' + ', '.join(f"{r.primary_type}({r['count']:,})" for _, r in top5.iterrows()))

Top 5 crímenes por año:
  2017 [PRE]: THEFT(64,386), BATTERY(49,239), CRIMINAL DAMAGE(29,045), DECEPTIVE PRACTICE(19,742), ASSAULT(19,306)
  2018 [PRE]: THEFT(65,290), BATTERY(49,832), CRIMINAL DAMAGE(27,823), ASSAULT(20,407), DECEPTIVE PRACTICE(19,927)
  2019 [PRE]: THEFT(62,497), BATTERY(49,522), CRIMINAL DAMAGE(26,682), ASSAULT(20,623), DECEPTIVE PRACTICE(19,190)
  2020 [DURANTE]: BATTERY(41,515), THEFT(41,344), CRIMINAL DAMAGE(24,878), DECEPTIVE PRACTICE(18,519), ASSAULT(18,258)
  2021 [DURANTE]: THEFT(40,819), BATTERY(40,472), CRIMINAL DAMAGE(25,096), ASSAULT(20,343), DECEPTIVE PRACTICE(17,770)
  2022 [DURANTE]: THEFT(54,890), BATTERY(40,949), CRIMINAL DAMAGE(27,248), MOTOR VEHICLE THEFT(21,466), ASSAULT(20,809)
  2023 [POST]: THEFT(57,469), BATTERY(44,223), CRIMINAL DAMAGE(30,087), MOTOR VEHICLE THEFT(29,251), ASSAULT(22,626)
  2024 [POST]: THEFT(59,956), BATTERY(46,004), CRIMINAL DAMAGE(28,497), ASSAULT(23,400), MOTOR VEHICLE THEFT(21,633)
  2025 [POST]: THEFT(21,054), BATTERY(1

---
## CRUD 3 — READ: Tasa de arresto por tipo de crimen y era COVID

In [6]:
arrest_by_era = (
    df.groupby(['primary_type', 'covid_era'])
      .agg({'arrest': 'mean', 'unique_key': 'count'})
      .rename(columns={'arrest': 'arrest_rate', 'unique_key': 'total_crimes'})
      .compute()
      .reset_index()
)
arrest_by_era['arrest_rate_pct'] = (arrest_by_era['arrest_rate'] * 100).round(2)
result = arrest_by_era[['primary_type', 'covid_era', 'total_crimes', 'arrest_rate_pct']]\
         .sort_values(['primary_type', 'covid_era'])

print('Variación de tasa de arresto por era (top 8 tipos de crimen más frecuentes):')
top_types = result.groupby('primary_type')['total_crimes'].sum().nlargest(8).index
pivot = result[result['primary_type'].isin(top_types)]\
        .pivot(index='primary_type', columns='covid_era', values='arrest_rate_pct')
print(pivot.to_string())

to_bigquery(result, 'arrest_rate_by_type')

Variación de tasa de arresto por era (top 8 tipos de crimen más frecuentes):
covid_era            DURANTE   POST    PRE
primary_type                              
ASSAULT                10.38  10.56  17.52
BATTERY                15.17  16.24  20.72
CRIMINAL DAMAGE         3.83   3.52   6.11
DECEPTIVE PRACTICE      1.71   3.64   4.49
MOTOR VEHICLE THEFT     3.44   2.76   6.55
OTHER OFFENSE          13.00  18.36  21.55
ROBBERY                 6.42   5.63   8.24
THEFT                   4.66   6.43   9.85


/tmp/ipykernel_38233/2663512432.py:16: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  df.to_gbq(destination_table=f'{DATASET_ID}.{table_name}',


  → BigQuery: my-first-project-492901.chicago_crimes_results.arrest_rate_by_type  (95 filas)


---
## CRUD 4 — UPDATE: Añadir `covid_era` y `severity_category`

In [7]:
VIOLENT_FBI = {'01A','01B','02','03','04A','04B','05','06','07','09'}
MEDIUM_FBI  = {'08A','08B','10','11','12'}

def assign_severity(code):
    if code in VIOLENT_FBI: return 'HIGH'
    if code in MEDIUM_FBI:  return 'MEDIUM'
    return 'LOW'

df['severity_category'] = df['fbi_code'].map(assign_severity, meta=('severity_category', 'object'))

severity = (
    df.groupby(['covid_era', 'severity_category'])
      .size()
      .reset_index()
      .rename(columns={0: 'count'})
      .compute()
)
era_totals = severity.groupby('covid_era')['count'].transform('sum')
severity['percentage'] = (severity['count'] / era_totals * 100).round(2)

print('Distribución de severidad por era COVID:')
pivot_sev = severity.pivot(index='severity_category', columns='covid_era', values='percentage')
print(pivot_sev[['PRE','DURANTE','POST']].to_string())

to_bigquery(severity, 'severity_distribution')

Distribución de severidad por era COVID:
covid_era            PRE  DURANTE   POST
severity_category                       
HIGH               15.19    17.71  20.06
LOW                55.88    51.56  51.82
MEDIUM             28.93    30.73  28.12


/tmp/ipykernel_38233/2663512432.py:16: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  df.to_gbq(destination_table=f'{DATASET_ID}.{table_name}',


  → BigQuery: my-first-project-492901.chicago_crimes_results.severity_distribution  (9 filas)


---
## CRUD 5 — DELETE: Eliminar registros sin coordenadas geográficas

In [8]:
missing_by_era = (
    df[df['latitude'].isna() | df['longitude'].isna()]
      .groupby('covid_era')
      .size()
      .compute()
      .rename('sin_coordenadas')
)
total_by_era_raw = df.groupby('covid_era').size().compute().rename('total')
coord_report = pd.concat([total_by_era_raw, missing_by_era], axis=1).fillna(0)
coord_report['pct_sin_coord'] = (coord_report['sin_coordenadas'] / coord_report['total'] * 100).round(3)

print('Registros sin coordenadas por era COVID:')
print(coord_report[['total','sin_coordenadas','pct_sin_coord']].loc[['PRE','DURANTE','POST']].to_string())

df_clean = df[~df['latitude'].isna() & ~df['longitude'].isna()]
n_clean  = len(df_clean.compute())
print(f'\nRegistros con coordenadas válidas: {n_clean:,}')

client.close()
cluster.close()
print('Cluster Dask cerrado. Resultados guardados en BigQuery.')

Registros sin coordenadas por era COVID:
            total  sin_coordenadas  pct_sin_coord
covid_era                                        
PRE        799839            12143          1.518
DURANTE    661583            15876          2.400
POST       611521             2390          0.391



Registros con coordenadas válidas: 2,042,534


Cluster Dask cerrado. Resultados guardados en BigQuery.
